# 이상 탐지

In [1]:
import os
import pandas as pd
import numpy as np
import hds
from plt_rcs import *

In [2]:
plt.rc(group='figure', figsize=(4, 4), dpi=120)

In [3]:
os.getcwd()

'/Users/taehyunan/Desktop/Repo/SeSAC/Study/sesac_ml_dl_study_repo/project/code/ml_part2'

In [4]:
os.chdir('../../data')

In [5]:
[i for i in os.listdir() if 'Sensor' in i and 'pkl' in i][0]

'Facility_Sensor_Raw.pkl'

In [6]:
objs = pd.read_pickle('Facility_Sensor_Raw.pkl')

In [7]:
globals().update(objs)

In [8]:
%whos

Variable    Type         Data/Info
----------------------------------
df          DataFrame    Shape: (10000, 7)
hds         module       <module 'hds' from '/opt/<...>ackages/hds/__init__.py'>
np          module       <module 'numpy' from '/op<...>kages/numpy/__init__.py'>
objs        dict         n=2
os          module       <module 'os' (frozen)>
pd          module       <module 'pandas' from '/o<...>ages/pandas/__init__.py'>
plt         module       <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
sns         module       <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>
train_num   DataFrame    Shape: (10000, 5)


In [9]:
df, train_num = df, train_num

## 타겟 벡터의 상대도수 확인

In [10]:
target_prop = df['Target'].value_counts(normalize=True)
target_prop
# Target
# 0    0.9661
# 1    0.0339
# Name: proportion, dtype: float64

Target
0    0.9661
1    0.0339
Name: proportion, dtype: float64

In [11]:
# 이상 탐지의 임계값을 활용
fail_rate = target_prop.loc[1]
fail_rate
# np.float64(0.0339)

np.float64(0.0339)

## 표준화 점수 기준 이상치 확인

In [12]:
train_num.head()

,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
Product ID,,,,,
M14860,298.1,308.6,1551,42.8,0
L47181,298.2,308.7,1408,46.3,3
L47182,298.1,308.5,1498,49.4,5
L47183,298.2,308.6,1433,39.5,7
L47184,298.2,308.7,1408,40.0,9


In [13]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# 표준화 객체 생성
scaler = StandardScaler()

In [ ]:
# 열별 표준화
X_scaled = pd.DataFrame(
    data=scaler.fit_transform(X=train_num),
    index=train_num.index,
    columns=train_num.columns
)

In [16]:
X_scaled.head()

,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
Product ID,,,,,
M14860,-0.952389,-0.947360,0.068185,0.282200,-1.695984
L47181,-0.902393,-0.879959,-0.729472,0.633308,-1.648852
L47182,-0.952389,-1.014761,-0.227450,0.944290,-1.617430
L47183,-0.902393,-0.947360,-0.590021,-0.048845,-1.586009
L47184,-0.902393,-0.879959,-0.729472,0.001313,-1.554588


In [17]:
X_scaled.describe().round(4)

,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
count,10000.0000,10000.0000,10000.0000,10000.0000,10000.0000
mean,-0.0000,0.0000,-0.0000,0.0000,0.0000
std,1.0001,1.0001,1.0001,1.0001,1.0001
min,-2.3523,-2.9020,-2.0682,-3.6301,-1.6960
25%,-0.8524,-0.8126,-0.6458,-0.6808,-0.8633
50%,0.0475,0.0637,-0.1996,0.0113,0.0008
75%,0.7475,0.7377,0.4084,0.6835,0.8491
max,2.2474,2.5575,7.5148,3.6729,2.2788


In [ ]:
# z-score 기준으로 열별 이상치 개수 확인
X_scaled.abs().apply(func= lambda x: x.ge(3).sum())
# AirTmp        0
# ProcTmp       0
# RotSpd      164
# Torque       25
# ToolWear      0
# dtype: int64

AirTmp        0
ProcTmp       0
RotSpd      164
Torque       25
ToolWear      0
dtype: int64

## 사분범위 기준 이상치 확인

In [ ]:
# 사분범위 기준 이상치 여부 반환 함수 정의
def iqr_outlier_count(x):
    q1, q3 = x.quantile(q=[0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier = x.lt(lower) | x.gt(upper)
    return outlier.sum()

In [ ]:
# 열별 이상치 개수 확인
X_scaled.apply(func= iqr_outlier_count)
# AirTmp        0
# ProcTmp       0
# RotSpd      418
# Torque       69
# ToolWear      0
# dtype: int64

AirTmp        0
ProcTmp       0
RotSpd      418
Torque       69
ToolWear      0
dtype: int64

## 마할라노비스 거리 기준 이상치 확인

In [21]:
from scipy.spatial.distance import mahalanobis

In [ ]:
# 열별 평균 벡터 생성
mean_vec = train_num.mean()
# AirTmp       300.00493
# ProcTmp      310.00556
# RotSpd      1538.77610
# Torque        39.98691
# ToolWear     107.95100
# dtype: float64

AirTmp       300.00493
ProcTmp      310.00556
RotSpd      1538.77610
Torque        39.98691
ToolWear     107.95100
dtype: float64

In [23]:
# 공분산 행렬 생성
cov_mat = train_num.cov()
cov_mat

,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
AirTmp,4.001035,2.600157,8.129957,-0.274736,1.763808
ProcTmp,2.600157,2.201467,5.127798,-0.207974,1.273840
RotSpd,8.129957,5.127798,32142.787047,-1563.910772,2.545883
Torque,-0.274736,-0.207974,-1563.910772,99.379640,-1.962568
ToolWear,1.763808,1.273840,2.545883,-1.962568,4051.850384


In [24]:
# 공분산 행렬의 역행렬 생성
cov_inv = np.linalg.inv(cov_mat)
cov_inv

array([[ 1.07571644e+00, -1.27026537e+00, -2.31072074e-04,
        -3.32219415e-03, -7.03812617e-05],
       [-1.27026537e+00,  1.95450924e+00,  1.60387180e-04,
         3.10136824e-03, -6.01081468e-05],
       [-2.31072074e-04,  1.60387180e-04,  1.32848846e-04,
         2.09032289e-03,  9.79167846e-07],
       [-3.32219415e-03,  3.10136824e-03,  2.09032289e-03,
         4.29549750e-02,  1.99635685e-05],
       [-7.03812617e-05, -6.01081468e-05,  9.79167846e-07,
         1.99635685e-05,  2.46859410e-04]])

In [25]:
# 마할라노비스 거리 생성
mnd = [mahalanobis(u=x, v=mean_vec, VI=cov_inv) for x in train_num.values]

In [26]:
from scipy import stats

In [ ]:
# 카이제곱 분포의 상위 1%에 해당하는 임계값 설정
mnd_threshold = stats.chi2.ppf(q=0.99, df=train_num.shape[1])
# 15.08627246938899

15.08627246938899


In [28]:
# 임계값의 양의 제곱근 확인
np.sqrt(mnd_threshold)
# np.float64(3.884105105347819)

np.float64(3.884105105347819)

In [29]:
# 마할라노비스 거리 기준 이상치 여부 생성
mnd_outlier = (mnd > np.sqrt(mnd_threshold))

In [30]:
# 마할라노비스 거리 기준 이상치 개수 확인
mnd_outlier.sum()
# np.int64(194)

np.int64(194)

In [ ]:
# 이상치 행 선택 후 Target의 상대도수 확인
df.loc[mnd_outlier, 'Target'].value_counts(normalize=True).sort_index()
# Target
# 0    0.587629
# 1    0.412371
# Name: proportion, dtype: float64

Target
0    0.587629
1    0.412371
Name: proportion, dtype: float64

## Hotelling's T2 기준 이상치 확인

In [32]:
from sklearn.decomposition import PCA

In [33]:
# 정상 데이터만 선택
X_normal = X_scaled.loc[df['Target'].eq(0)]

In [34]:
# 누적 분산 비율이 90%인 PCA 모델 생성
model_pca = PCA(n_components=0.90)

In [35]:
# 정상 데이터로 PCA 모델 학습
model_pca.fit(X=X_normal)

,n_components,0.9
,copy,True
,whiten,False
,svd_solver,'auto'
,tol,0.0
,iterated_power,'auto'
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,None


In [36]:
# 전체 데이터로 주성분 점수 행렬 생성
pca_score = model_pca.transform(X=X_scaled)

In [ ]:
# PCA 모델의 누적 분산 비율 확인
model_pca.explained_variance_ratio_.cumsum()
# array([0.39825224, 0.74785604, 0.95356962])

array([0.39825224, 0.74785604, 0.95356962])

In [38]:
# 각 주성분의 고유값 계산
lambdas = model_pca.explained_variance_

In [39]:
# 각 관측값별 Hotelling's T2 계산
ht2 = (pca_score ** 2 / lambdas).sum(axis=1)

In [40]:
# 주성분 개수 할당
m = model_pca.n_components_
# np.int64(3)

In [41]:
# 정상 데이터 개수 할당
n = X_normal.shape[0]
# 9661

In [42]:
# F 분포의 1%에 해당하는 임계값 계산
f_critical = stats.f.ppf(q=0.99, dfn=m, dfd=n-m)
# np.float64(3.783648204590835)

In [43]:
# F 분포를 이용해 Hotelling's T2 임계값 계산
ht2_threshold = m * (n - 1) / (n - m) * f_critical
# np.float64(11.353295192487305)

In [44]:
# Hotelling's T2 기준으로 이상치 여부 생성
ht2_outlier = (ht2 > ht2_threshold)

In [45]:
# Hotelling's T2 기준으로 이상치 개수 확인
ht2_outlier.sum()
# np.int64(139)

np.int64(139)

In [46]:
# 이상치 행을 선택하고 Target의 상대도수를 확인
df.loc[ht2_outlier, 'Target'].value_counts(normalize=True).sort_index()
# Target
# 0    0.741007
# 1    0.258993
# Name: proportion, dtype: float64

Target
0    0.741007
1    0.258993
Name: proportion, dtype: float64

## SPE 기준 이상치 확인

In [47]:
# PCA 모델로 전체 데이터 복원
X_hat = model_pca.inverse_transform(pca_score)

In [48]:
# 실제값과 복원 값의 잔차 계산
residual = X_scaled - X_hat

In [49]:
# 각 관측값별 SPE 계산
spe = (residual ** 2).sum(axis=1)

In [50]:
# 실제 고장 비율 기준 SPE 임계값 설정
spe_threshold = np.quantile(a=spe, q=1-fail_rate)

In [51]:
# SPE 기준 이상치 여부 생성
spe_outlier = (spe > spe_threshold)

In [52]:
# 이상치 개수 확인
spe_outlier.sum()
# np.int64(339)

np.int64(339)

In [53]:
# 이상치 행 선택 후 Taeget의 상대도수 확인
df.loc[spe_outlier, 'Target'].value_counts(normalize=True).sort_index()
# Target
# 0    0.666667
# 1    0.333333
# Name: proportion, dtype: float64

Target
0    0.666667
1    0.333333
Name: proportion, dtype: float64

## Isolation Forest 기준 이상치 확인

In [54]:
from sklearn.ensemble import IsolationForest

In [55]:
# Isolation Forest 모델 생성
iso = IsolationForest(
    n_estimators=100,
    contamination=fail_rate,
    random_state=0
)

In [56]:
# 모델 학습
iso.fit(X=train_num)

,n_estimators,100
,max_samples,'auto'
,contamination,np.float64(0.0339)
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,0
,verbose,0
,warm_start,False


In [57]:
# 관측값별 예측값 생성
iso_pred = iso.predict(X=train_num)

In [58]:
# Isolation Forest 기준으로 이상치 여부 생성
iso_outlier = (iso_pred == -1)

In [ ]:
# 이상치 개수 확인
iso_outlier.sum()
# np.int64(339)

np.int64(339)

In [60]:
# 이상치 행 선택 후 Taeget의 상대도수 확인
df.loc[iso_outlier, 'Target'].value_counts(normalize=True).sort_index()
# Target
# 0    0.855457
# 1    0.144543
# Name: proportion, dtype: float64

Target
0    0.855457
1    0.144543
Name: proportion, dtype: float64

## One-class SVM 기준 이상치 확인

In [61]:
from sklearn.svm import OneClassSVM

In [62]:
# One-class SVM 모델 생성
ocs = OneClassSVM(nu=fail_rate)

In [63]:
# 표준화된 데이터로 모델 학습
ocs.fit(X=X_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,nu,np.float64(0.0339)
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [64]:
# 관측값별 예측값 생성
ocs_pred = ocs.predict(X=X_scaled)

In [65]:
# 이상치 여부 생성
ocs_outlier = (ocs_pred == -1)

In [66]:
# 이상치 개수 확인
ocs_outlier.sum()
# np.int64(340)

np.int64(340)

In [67]:
# 이상치 행 선택 후 Taeget의 상대도수 확인
df.loc[ocs_outlier, 'Target'].value_counts(normalize=True).sort_index()
# Target
# 0    0.711765
# 1    0.288235
# Name: proportion, dtype: float64

Target
0    0.711765
1    0.288235
Name: proportion, dtype: float64

## Autoencoder

In [68]:
os.getcwd()

'/Users/taehyunan/Desktop/Repo/SeSAC/Study/sesac_ml_dl_study_repo/project/data'

In [69]:
os.chdir('../code/ml_part2')

In [70]:
[i for i in os.listdir() if '.py' in i]

['aed_anomaly.py', 'plt_rcs.py']

In [71]:
# from aed_anomaly import AEAnomalyDetector

In [72]:
# Autoencoder 기반 이상 탐지 객체 생성
# aed = AEAnomalyDetector(input_dim=X_scaled.shape[1], verbose=True)

In [73]:
# 정상 데이터로 신경망 모델 학습
# aed.fit(X=X_normal.values)

In [74]:
# 전체 데이터의 관측값별 재구성 오차 계산
# recon_error = aed.recon_error(X=X_scaled.values)

In [ ]:
# 오차 임계값 계산
# aed_threshold = np.quantile(a=recon_error, q=1-fail_rate)

In [ ]:
# 이상치 여부 생성
# aed_outlier = (recon_error > aed_threshold)

In [ ]:
# 이상치 개수 확인
# aed_outlier.sum()

In [78]:
# 이상치 행 선택 후 Taeget의 상대도수 확인
# df.loc[aed_outlier, 'Target'].value_counts(normalize=True).sort_index()

## 이상 탐지 기법별 이상 점수 추가

In [79]:
df['mnd_score'] = mnd
df['ht2_score'] = ht2
df['spe_score'] = spe
df['iso_score'] = -iso.decision_function(X=train_num)
df['ocs_score'] = -ocs.score_samples(X=X_scaled)

In [80]:
df.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear,Target,mnd_score,ht2_score,spe_score,iso_score,ocs_score
Product ID,,,,,,,,,,,,
M14860,2,298.1,308.6,1551,42.8,0,0,2.072996,3.776023,0.066992,-0.087002,-29.314971
L47181,1,298.2,308.7,1408,46.3,3,0,2.007077,3.992630,0.004252,-0.108508,-29.951258
L47182,1,298.1,308.5,1498,49.4,5,0,2.454524,3.943089,0.264745,-0.099779,-29.489676
L47183,1,298.2,308.6,1433,39.5,7,0,2.255281,3.412861,0.196973,-0.120025,-29.170037
L47184,1,298.2,308.7,1408,40.0,9,0,2.336999,3.314390,0.256856,-0.118840,-28.921655


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, M14860 to M24859
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Type       10000 non-null  int64  
 1   AirTmp     10000 non-null  float64
 2   ProcTmp    10000 non-null  float64
 3   RotSpd     10000 non-null  int64  
 4   Torque     10000 non-null  float64
 5   ToolWear   10000 non-null  int64  
 6   Target     10000 non-null  int64  
 7   mnd_score  10000 non-null  float64
 8   ht2_score  10000 non-null  float64
 9   spe_score  10000 non-null  float64
 10  iso_score  10000 non-null  float64
 11  ocs_score  10000 non-null  float64
dtypes: float64(8), int64(4)
memory usage: 1.2+ MB


## 특성 행렬과 타겟 벡터로 분리

In [83]:
yvar = 'Target'
X = df.drop(columns=yvar)
y = df[yvar].copy()

In [84]:
from sklearn.model_selection import train_test_split

In [85]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [88]:
os.getcwd()

'/Users/taehyunan/Desktop/Repo/SeSAC/Study/sesac_ml_dl_study_repo/project/code/ml_part2'

In [89]:
os.chdir('../../data')

In [91]:
pd.to_pickle(
    obj={
        'X_train': X_train,
        'X_valid': X_valid,
        'y_train': y_train,
        'y_valid': y_valid
    }, 
    filepath_or_buffer='Facility_Sensor_Out.pkl'
)